# Norman 2019 validation

In [1]:
import os
import sys
# os.chdir('/home/mohsen/projects/cpa/')
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [2]:
import cpa
import scanpy as sc

Global seed set to 0


In [3]:
sc.settings.set_figure_params(dpi=150)

In [4]:
data_dir = "/data/users/wergillius/CPA+GEARS/cpa_datasets"
data_path = 'Norman2019_prep_new.h5ad'

In [6]:
adata = sc.read(os.path.join(data_dir, data_path))
adata

AnnData object with n_obs × n_vars = 108497 × 5000
    obs: 'cov_drug_dose_name', 'dose_val', 'control', 'condition', 'guide_identity', 'drug_dose_name', 'cell_type', 'split', 'split1', 'split2', 'split3', 'split4', 'split5', 'split6', 'split7', 'split8', 'split9', 'split10', 'split11', 'split12', 'split13', 'split14', 'split15', 'split16', 'split17', 'split18', 'split19', 'split20', 'split21', 'split22', 'split23', 'split24', 'split25'
    var: 'gene_symbols', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'rank_genes_groups_cov'
    layers: 'counts'

In [15]:
adata.obs['guide_identity'].nunique()

289

# scVI 

In [39]:
import torch
from torch import nn
from torch.distributions import LogNormal, NegativeBinomial, Normal, kl_divergence

import scvi
from scvi import REGISTRY_KEYS
from scvi.module.base import BaseModuleClass, LossRecorder, auto_move_data

In [30]:
# try the vanilla scvi
scvi.model.SCVI.setup_anndata(
    adata=adata,
    layer='counts',
    categorical_covariate_keys = ['cell_type']
)

In [31]:
model = scvi.model.SCVI(adata,
                        gene_likelihood='nb'
                       )

In [44]:
model.module

VAE(
  (z_encoder): Encoder(
    (encoder): FCLayers(
      (fc_layers): Sequential(
        (Layer 0): Sequential(
          (0): Linear(in_features=5000, out_features=128, bias=True)
          (1): BatchNorm1d(128, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): None
          (3): ReLU()
          (4): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (mean_encoder): Linear(in_features=128, out_features=10, bias=True)
    (var_encoder): Linear(in_features=128, out_features=10, bias=True)
  )
  (l_encoder): Encoder(
    (encoder): FCLayers(
      (fc_layers): Sequential(
        (Layer 0): Sequential(
          (0): Linear(in_features=5000, out_features=128, bias=True)
          (1): BatchNorm1d(128, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): None
          (3): ReLU()
          (4): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (mean_encoder): Linear(in_features=128, out_features=1, bias=True)
 

In [33]:
model.train(train_size=0.8, validation_size=0.1,
            early_stopping=True
           )

GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [1]


Epoch 74/74: 100%|██████████| 74/74 [18:00<00:00, 14.60s/it, loss=1.13e+03, v_num=1]


In [34]:
save_path = os.path.join(os.path.dirname(data_dir), 
                         "cpa_pth", 
                         "Norman_scvi_negabino_loss.pth")

model.save(save_path)

```python
save_path = os.path.join(os.path.dirname(data_dir), "cpa_pth", "Norman_scvi_default.pth")

model.save(save_path)
```

In [35]:
from scvi.nn import FCLayers
from torch.nn.modules import activation

In [37]:
class Exponential_act(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self,x):
        return torch.exp(x)

In [41]:
# my simple VAE
class my_VAE(BaseModuleClass):
    def __init__(self, n_input, n_latent):
        super().__init__()
        # z encoder
        self.var_encoder = FCLayers(n_in=n_input, 
                                    n_out=n_latent, 
                                    activation_fn=Exponential_act
                                   )
        self.mean_encoder = FCLayers(n_in=n_input, 
                                     n_out=n_latent, 
                                     use_activation=False
                                    )
        # size encoder
        self.size_var_encoder = FCLayers(n_in=n_input, 
                                    n_out=1, 
                                    use_activation=False
                                   )
        self.size_mean_encoder = FCLayers(n_in=n_input, 
                                     n_out=1, 
                                     use_activation=False
                                    )
        # dispersion : the dimension should be the same as the n_var
        self.log_theta = nn.Parameter(torch.randn(n_input))
        
        # decoder
        self.decoder = FCLayers(n_in=n_latent,
                                n_out=n_input,
                                activation_fn=activation.Softmax
                               )
    
    def _get_inference_input(self, tensors):
        x = tensors[_CONSTANTS.X_KEY]
        input_dict = dict(x=x)
        return input_dict
    
    @auto_move_data
    def inference(self,x):
        """
        High level inference method.

        Runs the inference (encoder) model.
        """
        # log input !!
        x_ = torch.log(x + 1)
        
        # get mean and var
        qz_mean = self.mean_encoder(x_)
        qz_var = self.var_encoder(x_)
        # reparameterize
        prior = Normal(loc=qz_mean, scale=torch.sqrt(qz_mean))
        z_n = prior.rsample()
        
        # do the same thing for size factor
        l_mean = self.size_mean_encoder(x_)
        l_var = self.size_var_encoder(x_)
        # sample ln
        l_n = LogNormal(l_mean, torch.sqrt(l_var)).rsample()
        
        # the output dict
        outputs = dict(
            qz_m=qz_mean, qz_v=qz_var, z=z_n,
            l_m=l_mean, l_v=l_var, l=l_n
                      )
        return outputs
        
    def _get_generative_input(self, tensors, inference_output):
        z = inference_output["z"]
        l = inference_output["l"]
        
        return {"z":z, "library":l}
        
        
    @auto_move_data
    def generative(self, z, library):
        
        # get the normalized mean of negetive binomial 
        px_scale = self.decoder(z)
        
        # timing the library size
        px_rate = px_scale * library
        
        # get the dispersion
        theta = torch.exp(self.log_theta)
        
        NegBino_params = {'px_scale':px_scale,
                          'px_rate':px_rate,
                          'theta':log_theta
                         }
        return NegBino_params
    
    def loss(self, tensors, inference_output, generative_output):
        """
        # here, we would like to form the ELBO. There are two terms:
        #   1. one that pertains to the likelihood of the data
        #   2. one that pertains to the variational distribution
        """
        # so we extract all the required information
        x = tensors[REGISTRY_KEYS.X_KEY]
        qz_m = inference_output['qz_m']
        qz_v = inference_output['qz_v']
        z = inference_output['z']
        library = inference_output['l']
        
        px_scale = generative_output['px_scale']
        px_rate = generative_output['px_rate']
        theta = generative_output['theta']
        
        # term 1 : compute the likelihood
        # the pytorch NB distribution uses a different parameterization
        # so we must apply a quick transformation 
        # (included in scvi-tools, but here we use the pytorch code)
        nb_logit = (px_rate + 1e-4).log() - (theta + 1e-4).log()
        px = NegativeBinomial(total_count=theta, logits=nb_logits)
        log_lik = px.log_prob(x).sum(dim=-1)
        
        # term 2 : KL divergence 
        # standard gaussian
        prior = Normal(torch.zeros_like(qz_m), torch.ones_like(qz_v))
        posterior = Normal(qz_m, qz_v)
        kld = kl_divergence(prior, posterior).sum(dim=1)
        
        elbo = log_lik - kld
        loss = torch.mean(-elbo)
        return LossRecorder(loss, -log_like, kld, 0.0)